# TP2 — Submit to the Weather Prediction Competition

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/racousin/L2Math/blob/main/session3/tp2_correction.ipynb)

## Objective

Take the 3 models you trained in TP1, wrap them into an `Agent` class, test it locally (reproducing the platform's evaluation), and submit to the live competition.

### Competition Info

| | |
|---|---|
| **Enroll** | [ml-arena.com/enroll/6db4734d376f7b64249f503f140db96f](https://ml-arena.com/enroll/6db4734d376f7b64249f503f140db96f) |
| **Competition page** | [ml-arena.com/viewcompetition/26](https://ml-arena.com/viewcompetition/26) |
| **Leaderboard assessed** | 30-day rolling average — last submission April 10 / assessed May 15, 2026 |
| **Presentation** | ~April 10, 2026 |
| **Submission** | Web interface — upload `agent.py` + `model_temperature.pkl` + `model_wind_speed.pkl` + `model_rain.pkl` |

### Roadmap

| Section | What you do |
|---------|-------------|
| 1. Enroll & Understand | Join the competition, understand input/output |
| 2. Build the Agent | Bridge TP1 features → Agent class |
| 3. Test Locally | Reproduce platform evaluation with real data |
| 4. Submit | Upload to the competition |
| 5. Ideas for Improvement | Guidance for the next month |

---
## 1. Setup

In [1]:
!pip install scikit-learn pandas numpy joblib


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import numpy as np
import pandas as pd
import joblib

---
## 2. Enroll & Understand the Competition

### Exercise 1.1 — Enroll

1. Go to the enrollment link: [ml-arena.com/enroll/6db4734d376f7b64249f503f140db96f](https://ml-arena.com/enroll/6db4734d376f7b64249f503f140db96f)
2. Create an account (or log in)
3. Explore the competition page: leaderboard, description, rules

### Exercise 1.2 — Understand the input/output

Your agent receives a 3D numpy array `X_test` of shape `(20, 24, 8)` and must return a prediction of shape `(3,)`.

**Input: `X_test` — shape `(20, 24, 8)`**

| Axis | Size | Meaning |
|------|------|---------|
| 0 | 20 | Cities (alphabetical order) |
| 1 | 24 | Hours of history (oldest → newest) |
| 2 | 8 | Weather features |

**20 cities** (alphabetical — axis 0):

| Index | City | Index | City |
|-------|------|-------|------|
| 0 | Amsterdam | 10 | Köln |
| 1 | Barcelona | 11 | London |
| 2 | Birmingham | 12 | Manchester |
| 3 | Brussels | 13 | Marseille |
| 4 | Copenhagen | 14 | Milan |
| 5 | Dortmund | 15 | Munich |
| 6 | Dublin | 16 | **Paris** |
| 7 | Düsseldorf | 17 | Rotterdam |
| 8 | Essen | 18 | Stuttgart |
| 9 | Frankfurt am Main | 19 | Turin |

**8 features** (axis 2):

| Index | Feature | Unit |
|-------|---------|------|
| 0 | temperature | °C |
| 1 | rain | mm |
| 2 | wind_speed | m/s |
| 3 | wind_direction | degrees |
| 4 | humidity | % |
| 5 | clouds | % |
| 6 | visibility | m |
| 7 | snow | mm |

**Output: prediction — shape `(3,)`**

| Index | Target | Unit |
|-------|--------|------|
| 0 | temperature | °C |
| 1 | wind_speed | m/s |
| 2 | rain | mm |

**Question:** Where is Paris in the input array? What is `X_test[16, -1, 0]`?

**Answer:** Paris is at index 16 (alphabetical order). `X_test[16, -1, 0]` is the most recent temperature reading for Paris (last hour, feature index 0 = temperature).

### Exercise 1.3 — Understand the scoring metric

The competition uses **Negated Normalized MAE**:

```
score = -mean(|pred - true| / std)
```

where:
- `std_temperature = 7.49 °C`
- `std_wind_speed = 5.05 m/s`
- `std_rain = 0.40 mm`

**Question:** Why is rain the hardest to predict well? (Think about what happens when you're off by 1mm on rain vs 1°C on temperature.)

**Answer:** Rain has the smallest std (0.40 mm), so errors are amplified the most after normalization. Being off by 1mm on rain costs `1/0.40 = 2.5` in normalized error, while being off by 1°C on temperature only costs `1/7.49 = 0.13`. Rain errors dominate the score.

---
## 3. Build the Agent Class

### Exercise 2.1 — Understand the Agent contract

The platform calls your agent like this:

```python
agent = Agent()                     # loads model in __init__
prediction = agent.predict(X_test)  # X_test: (20, 24, 8) → returns (3,)
```

The baseline agent (last known Paris values):

```python
class Agent:
    def predict(self, X_test):
        paris = X_test[16]  # (24, 8)
        return np.array([paris[-1, 0], paris[-1, 2], paris[-1, 1]])
        # [last temperature, last wind_speed, last rain]
```

Note the output order: `[temperature, wind_speed, rain]` — but in the input features, rain is index 1 and wind_speed is index 2.

### Exercise 2.2 — Bridge TP1 → TP2: from DataFrame to 3D array

In TP1, you built features from a DataFrame (pivot table with columns like `Paris_temperature`, `London_rain`, etc.).

On the platform, you get a **3D numpy array** `(20, 24, 8)`. You need to extract the same features.

Here's the mapping:

```python
PARIS_IDX = 16

# TP1: df['Paris_temperature']  →  TP2: X_test[16, :, 0]
# TP1: df['London_rain']        →  TP2: X_test[11, :, 1]
# TP1: df['Paris_temperature_lag3']  →  TP2: X_test[16, -3, 0]
# TP1: df['Munich_wind_speed']  →  TP2: X_test[15, :, 2]
```

Write a function `extract_features(X_test)` that extracts the same features you used in TP1 from the 3D array and returns a 1D numpy array.

*Hint:* Look at what features your TP1 model uses. A simple starting point: last-hour values from all cities + Paris lag features.

In [3]:
CITIES = [
    "Amsterdam", "Barcelona", "Birmingham", "Brussels", "Copenhagen",
    "Dortmund", "Dublin", "Düsseldorf", "Essen", "Frankfurt am Main",
    "Köln", "London", "Manchester", "Marseille", "Milan", "Munich",
    "Paris", "Rotterdam", "Stuttgart", "Turin",
]
PARIS_IDX = 16

# Feature indices in X_test axis 2
TEMP_IDX = 0
RAIN_IDX = 1
WIND_IDX = 2
WIND_DIR_IDX = 3
HUMIDITY_IDX = 4
CLOUDS_IDX = 5
VISIBILITY_IDX = 6
SNOW_IDX = 7

# Alphabetical order to match pivot_table column sorting from TP1:
# clouds, humidity, rain, snow, temperature, wind_direction, wind_speed
FEATURE_INDICES = [CLOUDS_IDX, HUMIDITY_IDX, RAIN_IDX, SNOW_IDX, TEMP_IDX, WIND_DIR_IDX, WIND_IDX]

def extract_features(X_test):
    """Extract features from (20, 24, 8) array → 1D feature vector.
    Matches the features used to train model.pkl in TP1 correction:
    - All cities × all weather features (last hour) — alphabetical order matching pivot columns
    - Paris lag features (T-1 to T-6) for temperature, wind_speed, rain
    - Time features: sin_hour, cos_hour
    """
    features = []

    # 1. All cities, last hour, 7 weather features (alphabetical order)
    # This matches the pivot table columns: clouds_Amsterdam, ..., wind_speed_Turin
    for feat_idx in FEATURE_INDICES:
        for city_idx in range(20):
            features.append(X_test[city_idx, -1, feat_idx])

    # 2. Paris lag features: T-1 to T-6 for temperature, wind_speed, rain
    for lag in range(1, 7):
        features.append(X_test[PARIS_IDX, -lag, TEMP_IDX])   # temperature_Paris_lag{lag}
        features.append(X_test[PARIS_IDX, -lag, WIND_IDX])   # wind_speed_Paris_lag{lag}
        features.append(X_test[PARIS_IDX, -lag, RAIN_IDX])   # rain_Paris_lag{lag}

    # 3. Time features — not available from X_test alone
    # sin_hour and cos_hour (use 0 as placeholder)
    features.extend([0, 0])

    return np.array(features)

print(f"Feature vector length: {len(extract_features(np.random.randn(20, 24, 8)))}")

Feature vector length: 160


### Exercise 2.3 — Write the full Agent class

Combine `extract_features` and your 3 models into a complete Agent class.

*Hint:* Load `model_temperature.pkl`, `model_wind_speed.pkl`, `model_rain.pkl` in `__init__`, extract features and predict each target in `predict()`.

In [4]:
class Agent:
    def __init__(self):
        self.model_temperature = joblib.load('model_temperature.pkl')
        self.model_wind_speed = joblib.load('model_wind_speed.pkl')
        self.model_rain = joblib.load('model_rain.pkl')

    def predict(self, X_test):
        """
        X_test: np.ndarray of shape (20, 24, 8)
        Returns: np.ndarray of shape (3,) — [temperature, wind_speed, rain]
        """
        features = extract_features(X_test)
        X = features.reshape(1, -1)
        temperature = self.model_temperature.predict(X)[0]
        wind_speed = self.model_wind_speed.predict(X)[0]
        rain = self.model_rain.predict(X)[0]
        return np.array([temperature, wind_speed, rain])

---
## 4. Test Agent Locally

Before submitting, let's test the agent with real data to make sure it works correctly.

### Exercise 3.1 — Build a real `(20, 24, 8)` window from the CSV

Load `weather_paris_20cities.csv` and build a test window:
1. Pick 24 consecutive hours (e.g., 2025-06-15 00:00 to 2025-06-15 23:00)
2. For each hour, get the 20 cities (sorted alphabetically)
3. Extract the 8 features: temperature, rain, wind_speed, wind_direction, humidity, clouds, visibility (use 0 if missing), snow
4. Result: array of shape `(20, 24, 8)`

Also get the ground truth: Paris weather at T+6h (6 hours after the last hour in the window).

*Hint:*
```python
feature_cols = ['temperature', 'rain', 'wind_speed', 'wind_direction', 
                'humidity', 'clouds', 'visibility', 'snow']
# Note: visibility is not in our CSV — use 0 as placeholder (index 6)
```

In [5]:
df = pd.read_csv('weather_paris_20cities.csv', parse_dates=['timestamp'])

# Features in the order expected by the competition (8 features)
csv_feature_cols = ['temperature', 'rain', 'wind_speed', 'wind_direction',
                    'humidity', 'clouds', 'snow']  # 7 cols in our CSV (no visibility)

def build_window(df, start_time):                                                     
  """Build a (20, 24, 8) window and ground truth from CSV data."""                
  hours = pd.date_range(start_time, periods=24, freq='h')                           
  gt_time = hours[-1] + pd.Timedelta(hours=6)                                       
                                                                                    
  cities_sorted = sorted(df['city_name'].unique())                                  
  city_to_idx = {c: i for i, c in enumerate(cities_sorted)}                       
                                                                                    
  csv_to_comp = {                                                                   
      'temperature': 0, 'rain': 1, 'wind_speed': 2, 'wind_direction': 3,            
      'humidity': 4, 'clouds': 5, 'snow': 7                                         
  }                                                                                 
                                                                                    
  window = np.zeros((20, 24, 8))                                                    
  subset = df[df['timestamp'].isin(hours)]                                          
  for (ts, city), group in subset.groupby(['timestamp', 'city_name']):              
      h_idx = hours.get_loc(ts)                                                     
      c_idx = city_to_idx[city]                                                     
      row = group.iloc[0]                                                           
      for col, comp_idx in csv_to_comp.items():                                     
          val = row[col]                                                            
          window[c_idx, h_idx, comp_idx] = val if not pd.isna(val) else 0.0         
                                                                                    
  # Ground truth: Paris [temperature, wind_speed, rain] at T+6h                     
  gt_row = df[(df['timestamp'] == gt_time) & (df['city_name'] == 'Paris')].iloc[0]
  ground_truth = np.array([gt_row['temperature'], gt_row['wind_speed'],             
gt_row['rain']])                                                                      
                                                                                    
  return window, ground_truth   

# Build one test window
X_test, y_true = build_window(df, '2025-06-15 00:00:00+00:00')

print(f"X_test shape: {X_test.shape}")
print(f"Ground truth (temp, wind, rain): {y_true}")
print(f"\nParis last hour temp: {X_test[PARIS_IDX, -1, TEMP_IDX]:.1f} °C")
print(f"Paris last hour wind: {X_test[PARIS_IDX, -1, WIND_IDX]:.1f} m/s")
print(f"Paris last hour rain: {X_test[PARIS_IDX, -1, RAIN_IDX]:.1f} mm")

X_test shape: (20, 24, 8)
Ground truth (temp, wind, rain): [12.5  6.4  0. ]

Paris last hour temp: 15.7 °C
Paris last hour wind: 6.2 m/s
Paris last hour rain: 0.0 mm


### Exercise 3.2 — Run the agent and check output

Instantiate your `Agent` and call `predict(X_test)`. Verify:
- Output shape is `(3,)`
- Values are reasonable (temperature in [-10, 45], wind_speed in [0, 30], rain in [0, 20])

*Hint:* `agent = Agent()`, `pred = agent.predict(X_test)`, `print(pred.shape, pred)`

In [6]:
agent = Agent()
pred = agent.predict(X_test)

print(f"Prediction shape: {pred.shape}")
print(f"Prediction: temp={pred[0]:.2f}°C, wind={pred[1]:.2f}m/s, rain={pred[2]:.3f}mm")
print(f"Ground truth: temp={y_true[0]:.2f}°C, wind={y_true[1]:.2f}m/s, rain={y_true[2]:.3f}mm")

# Sanity checks
assert pred.shape == (3,), f"Wrong shape: {pred.shape}, expected (3,)"
assert -30 < pred[0] < 50, f"Temperature out of range: {pred[0]}"
assert 0 <= pred[1] < 50, f"Wind speed out of range: {pred[1]}"
assert pred[2] >= 0, f"Rain should be non-negative: {pred[2]}"
print("\nAll sanity checks passed.")

Prediction shape: (3,)
Prediction: temp=25.44°C, wind=21.64m/s, rain=0.000mm
Ground truth: temp=12.50°C, wind=6.40m/s, rain=0.000mm

All sanity checks passed.


/Users/raphaelcousin/Library/Caches/pypoetry/virtualenvs/l2math-j4wtc_fC-py3.13/lib/python3.13/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but GradientBoostingRegressor was fitted with feature names
  warnings.warn(
/Users/raphaelcousin/Library/Caches/pypoetry/virtualenvs/l2math-j4wtc_fC-py3.13/lib/python3.13/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but GradientBoostingRegressor was fitted with feature names
  warnings.warn(
/Users/raphaelcousin/Library/Caches/pypoetry/virtualenvs/l2math-j4wtc_fC-py3.13/lib/python3.13/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but GradientBoostingRegressor was fitted with feature names
  warnings.warn(


### Exercise 3.3 — Reproduce platform evaluation

Compare your prediction with the actual Paris values at T+6h. Compute the Negated Normalized MAE — this is what you'll see on the leaderboard.

```python
std = np.array([7.49, 5.05, 0.40])  # [temperature, wind_speed, rain]
score = -np.mean(np.abs(prediction - ground_truth) / std)
```

Try with multiple windows to get a sense of your model's average performance.

*Hint:* Loop over several start times, build window + ground truth, predict, compute score, then average.

In [7]:
std = np.array([7.49, 5.05, 0.40])

# Evaluate on multiple windows spread across the year
test_starts = [
    '2025-03-15 06:00:00+00:00',
    '2025-05-01 12:00:00+00:00',
    '2025-06-15 00:00:00+00:00',
    '2025-08-20 18:00:00+00:00',
    '2025-10-10 06:00:00+00:00',
    '2025-11-25 12:00:00+00:00',
]

scores = []
for start in test_starts:
    X_w, y_w = build_window(df, start)
    pred_w = agent.predict(X_w)
    score = -np.mean(np.abs(pred_w - y_w) / std)
    scores.append(score)
    print(f"{start[:10]}: pred=[{pred_w[0]:6.2f}, {pred_w[1]:5.2f}, {pred_w[2]:5.3f}]  "
          f"true=[{y_w[0]:6.2f}, {y_w[1]:5.2f}, {y_w[2]:5.3f}]  score={score:.4f}")

print(f"\nAverage score: {np.mean(scores):.4f}")
print(f"Std of scores: {np.std(scores):.4f}")

2025-03-15: pred=[ 19.48, 21.25, 0.000]  true=[  6.80, 11.70, 0.000]  score=-1.1948
2025-05-01: pred=[ 24.41, 20.04, 0.000]  true=[ 28.50,  7.00, 0.000]  score=-1.0432
2025-06-15: pred=[ 25.44, 21.64, 0.000]  true=[ 12.50,  6.40, 0.000]  score=-1.5820
2025-08-20: pred=[ 24.75, 22.93, 0.000]  true=[ 15.90,  9.70, 0.000]  score=-1.2668
2025-10-10: pred=[ 23.85, 23.90, 0.000]  true=[ 16.10,  9.70, 0.000]  score=-1.2826
2025-11-25: pred=[ 25.34, 20.09, 0.000]  true=[  5.50,  7.20, 0.000]  score=-1.7340

Average score: -1.3506
Std of scores: 0.2349


/Users/raphaelcousin/Library/Caches/pypoetry/virtualenvs/l2math-j4wtc_fC-py3.13/lib/python3.13/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but GradientBoostingRegressor was fitted with feature names
  warnings.warn(
/Users/raphaelcousin/Library/Caches/pypoetry/virtualenvs/l2math-j4wtc_fC-py3.13/lib/python3.13/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but GradientBoostingRegressor was fitted with feature names
  warnings.warn(
/Users/raphaelcousin/Library/Caches/pypoetry/virtualenvs/l2math-j4wtc_fC-py3.13/lib/python3.13/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but GradientBoostingRegressor was fitted with feature names
  warnings.warn(
/Users/raphaelcousin/Library/Caches/pypoetry/virtualenvs/l2math-j4wtc_fC-py3.13/lib/python3.13/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature name

---
## 5. Submit via Web Interface

### Exercise 4.1 — Save your `agent.py` file

Your `agent.py` file must contain a class `Agent` with:
- `__init__(self)` — loads `model_temperature.pkl`, `model_wind_speed.pkl`, `model_rain.pkl`
- `predict(self, X_test)` — takes `(20, 24, 8)`, returns `(3,)`

A template is provided in `session3/agent.py`. Adapt it to match the features your model expects.

Save your final version:

In [8]:
agent_code = '''import numpy as np
import joblib

PARIS_IDX = 16

# Feature indices in X_test axis 2, ordered alphabetically to match pivot_table output
# pivot_table sorts values alphabetically: clouds, humidity, rain, snow, temperature, wind_direction, wind_speed
CLOUDS_IDX = 5
HUMIDITY_IDX = 4
RAIN_IDX = 1
SNOW_IDX = 7
TEMP_IDX = 0
WIND_DIR_IDX = 3
WIND_IDX = 2

FEATURE_INDICES = [CLOUDS_IDX, HUMIDITY_IDX, RAIN_IDX, SNOW_IDX, TEMP_IDX, WIND_DIR_IDX, WIND_IDX]


class Agent:
    def __init__(self):
        self.model_temperature = joblib.load("model_temperature.pkl")
        self.model_wind_speed = joblib.load("model_wind_speed.pkl")
        self.model_rain = joblib.load("model_rain.pkl")

    def predict(self, X_test):
        features = []

        # All cities, last hour, 7 weather features (alphabetical order to match pivot columns)
        for feat_idx in FEATURE_INDICES:
            for city_idx in range(20):
                features.append(X_test[city_idx, -1, feat_idx])

        # Paris lag features: T-1 to T-6 for temperature, wind_speed, rain
        for lag in range(1, 7):
            features.append(X_test[PARIS_IDX, -lag, TEMP_IDX])
            features.append(X_test[PARIS_IDX, -lag, WIND_IDX])
            features.append(X_test[PARIS_IDX, -lag, RAIN_IDX])

        # Time features: sin_hour, cos_hour (not available from X_test — use 0 as placeholder)
        features.extend([0, 0])

        X = np.array(features).reshape(1, -1)
        temperature = self.model_temperature.predict(X)[0]
        wind_speed = self.model_wind_speed.predict(X)[0]
        rain = self.model_rain.predict(X)[0]
        return np.array([temperature, wind_speed, rain])
'''

with open('agent.py', 'w') as f:
    f.write(agent_code)
print('agent.py saved!')
print(f'File size: {len(agent_code)} bytes')

agent.py saved!
File size: 1669 bytes


### Exercise 4.2 — Submit to the competition

1. Go to [ml-arena.com/viewcompetition/26](https://ml-arena.com/viewcompetition/26)
2. Click **"Submit Agent"** / choose a name
3. Upload your files:
   - `agent.py` — your Agent class
   - `model_temperature.pkl` — temperature model
   - `model_wind_speed.pkl` — wind speed model
   - `model_rain.pkl` — rain model
4. Choose runtime framework (sickit_learn)
4. **deployed**
5. Check your score on the leaderboard

If the status shows **"Failed"**, check the error message — common issues:
- Wrong output shape (must be `(3,)`)
- Import errors (only `numpy`, `pandas`, `scikit-learn`, `joblib` are available)
- Feature mismatch between training and prediction

---
## 6. Ideas for Improvement

You have **~1 month by team (2-3)** to iterate and climb the leaderboard. Here are ideas to explore:

### Rain is the hardest target
Rain has `std = 0.40 mm`, so being off by just 1mm costs `1/0.40 = 2.5` in normalized error. Temperature has `std = 7.49`, so 1°C off only costs `0.13`. Focus effort on rain prediction.

### Use neighboring cities as predictors
Weather systems move — typically west → east in Europe. London/Dublin weather a few hours ago can predict what's coming to Paris. Look at which cities are most correlated with Paris at different lags.

### Wind direction as a signal
Westerly winds (from the Atlantic) bring rain and mild temperatures. Easterly winds bring continental weather (cold/dry in winter, hot in summer). Wind direction from upstream cities is especially informative.

### Seasonal adaptation
The 30-day rolling leaderboard rewards models that adapt to the current season. A model trained on full-year data might underperform one tuned to the current month's patterns. Consider retraining or using seasonal features.

### Try different algorithms
- **Neural networks** — can capture complex temporal patterns (but need more data/tuning)
- **Ensemble of models** — average predictions from multiple approaches

### More feature ideas
- Rolling statistics (mean/std over last 6/12/24h)
- Differences (temperature change in last 3h)
- Cross-city features (temperature gradient west → east)
- Pressure proxy from humidity + clouds patterns

---
## 7. Summary

### What you did

| Step | What |
|------|------|
| **Understand** | Input `(20, 24, 8)` → output `(3,)`, scoring metric |
| **Build Agent** | Bridged TP1 features to 3D array extraction |
| **Test locally** | Built real windows, reproduced platform scoring |
| **Submit** | Uploaded to ml-arena.com |

### Deadlines

| Date | What |
|------|------|
| **Now** | First submission — make sure it runs! |
| **~April 10, 2026** | Presentation — explain your approach |
| **Last submission. April 10 / May 15, 2026** | 30-day rolling leaderboard assessed for final grade |

Good luck — and may the best model win!